In [1]:
# ── Imports ──
import pandas as pd
import numpy as np
import mlflow
import dagshub
from mlflow.tracking import MlflowClient

dagshub.init(
    repo_owner='lkhiz23',
    repo_name='House-Prices',
    mlflow=True
)

Accessing as lkhiz23

Initialized MLflow to track repo "lkhiz23/House-Prices"

Repository lkhiz23/House-Prices initialized!

In [14]:
model_name = "HousePrices_Best"
model_version = 3

model_uri = f"models:/{model_name}/{model_version}"
pipeline = mlflow.sklearn.load_model(model_uri)

print(f"✅ Model loaded: {model_name} v{model_version}")

✅ Model loaded: HousePrices_Best v3


In [15]:
# ── Load and prepare test data ──
test = pd.read_csv('data/test.csv')
print(f"Test shape: {test.shape}")

Test shape: (1459, 80)


In [17]:
# ── Apply same feature engineering as training ──
# Copy add_features function here
def add_features(df):
    df = df.copy()

    # Age features
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['RemodelAge'] = df['YrSold'] - df['YearRemodAdd']
    df['GarageAge'] = df['YrSold'] - df['GarageYrBlt']

    # Total area features
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['TotalPorchSF'] = (df['WoodDeckSF'] + df['OpenPorchSF'] +
                          df['EnclosedPorch'] + df['3SsnPorch'] +
                          df['ScreenPorch'])

    # Bathroom aggregation
    df['TotalBath'] = (df['FullBath'] + 0.5*df['HalfBath'] +
                       df['BsmtFullBath'] + 0.5*df['BsmtHalfBath'])

    # Binary features
    df['HasPool'] = (df['PoolArea'] > 0).astype(int)
    df['HasGarage'] = (df['GarageCars'] > 0).astype(int)
    df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)
    df['IsRemodeled'] = (df['YearBuilt'] != df['YearRemodAdd']).astype(int)

    # Interaction
    df['OverallQualXGrLivArea'] = df['OverallQual'] * df['GrLivArea']

    return df

In [20]:
# Correct columns that pipeline expects
numeric_cols_pipeline = ['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
                          'OverallCond', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2',
                          'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
                          'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath',
                          'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr',
                          'TotRmsAbvGrd', 'GarageCars', 'WoodDeckSF', 'OpenPorchSF',
                          'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'MiscVal',
                          'MoSold', 'YrSold', 'HouseAge', 'RemodelAge', 'GarageAge',
                          'TotalSF', 'TotalPorchSF', 'TotalBath', 'HasPool', 'HasGarage',
                          'HasFireplace', 'IsRemodeled', 'OverallQualXGrLivArea']

ordinal_cols_pipeline = ['HeatingQC', 'BsmtExposure', 'Functional', 'BsmtFinType1']

cat_cols_pipeline = ['LotShape', 'LotConfig', 'Neighborhood', 'Exterior1st',
                     'Exterior2nd', 'Heating', 'CentralAir', 'GarageFinish',
                     'Fence', 'SaleCondition']

all_cols_pipeline = numeric_cols_pipeline + ordinal_cols_pipeline + cat_cols_pipeline

test_fe = add_features(test)
test_final = test_fe[all_cols_pipeline]

print(f"test_final shape: {test_final.shape}")

test_final shape: (1459, 56)


In [21]:
# Predict
log_predictions = pipeline.predict(test_final)
predictions = np.expm1(log_predictions)
print(f"Sample predictions: {predictions[:5]}")

Sample predictions: [119012.21755646 156696.36944156 177785.58856851 198650.52872878
 191902.86914991]


In [22]:
# Generate submission file
submission = pd.DataFrame({
    'Id': test['Id'],
    'SalePrice': predictions
})

submission.to_csv('data/submission.csv', index=False)
print(f"✅ Submission saved!")
print(f"Shape: {submission.shape}")
print(submission.head())

✅ Submission saved!
Shape: (1459, 2)
     Id      SalePrice
0  1461  119012.217556
1  1462  156696.369442
2  1463  177785.588569
3  1464  198650.528729
4  1465  191902.869150
